# Multipole-to-local conversion (M2L)

## Purpose

M2L converts a source-centred multipole expansion into a local Taylor
expansion about a distant target centre. This is the central far-field
interaction operator in a classical FMM. The resulting coefficients describe
the incoming potential near the target centre; M2L alone does not evaluate a
field at a target particle.

## Mathematical definition

For $R=c_t-c_s$ and local multi-index $\beta$,

$$L_\beta \mathrel{+}= \sum_{|\alpha|\le p}
M_\alpha D_{\alpha+\beta}G(R).$$

Because $|\alpha+\beta|$ can reach $2p$, M2L internally needs Laplace
derivatives through order $2p$.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import cdfmm

try:
    from example_utils import (
        direct_fields,
        draw_box_3d,
        error_metrics,
        finish_3d_axes,
        local_fields,
        multipole_fields,
        new_3d_figure,
        nodes_at_level,
        plot_coefficients_by_degree,
        random_unit_vectors,
        relative_error,
        set_axes_equal,
        vec3_to_array,
    )
except ModuleNotFoundError:
    # This path is used when the kernel starts in the repository root.
    from examples.notebooks.example_utils import (
        direct_fields,
        draw_box_3d,
        error_metrics,
        finish_3d_axes,
        local_fields,
        multipole_fields,
        new_3d_figure,
        nodes_at_level,
        plot_coefficients_by_degree,
        random_unit_vectors,
        relative_error,
        set_axes_equal,
        vec3_to_array,
    )

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True})

## User parameters

In [ ]:
expansion_order = 4
n_sources = 60
source_box_half_width = 0.4
random_seed = 42
source_centre = np.array([0.0, 0.0, 0.0])
target_centre = np.array([4.0, 0.75, -0.5])

## Problem setup and operator evaluation

In [ ]:
rng = np.random.default_rng(random_seed)
source_positions = rng.uniform(
    -source_box_half_width,
    source_box_half_width,
    size=(n_sources, 3),
)
dipole_moments = rng.normal(size=(n_sources, 3))

multipole_coefficients = cdfmm.p2m_dipole(
    source_centre,
    source_positions,
    dipole_moments,
    order=expansion_order,
)
local_coefficients = cdfmm.m2l(
    multipole_coefficients,
    source_centre,
    target_centre,
    order=expansion_order,
)
multi_indices = cdfmm.multi_indices(expansion_order)

print(f"Expansion order: {expansion_order}")
print(f"Coefficient count: {len(local_coefficients)}")
print(f"Centre separation: {np.linalg.norm(target_centre - source_centre):.3f}")
print("First ten local coefficients:")
for beta, value in list(zip(multi_indices, local_coefficients))[:10]:
    print(f"  beta={tuple(beta)}  degree={np.sum(beta)}  L_beta={value: .8e}")

## Coefficient inspection

In [ ]:
figure, axes = plt.subplots(1, 2, figsize=(12, 4.5))
plot_coefficients_by_degree(
    axes[0],
    multipole_coefficients,
    expansion_order,
    "Source multipole coefficients",
)
plot_coefficients_by_degree(
    axes[1],
    local_coefficients,
    expansion_order,
    "Target local coefficients",
    colour="tab:green",
)
figure.tight_layout()

## Reference comparison at the local centre

In [ ]:
# At dx=0, the degree-one local coefficients give H=-grad(phi).
local_centre_result = cdfmm.l2p(
    local_coefficients,
    target_centre,
    target_centre,
    order=expansion_order,
    output="both",
)
direct_result = cdfmm.p2p_dipole_sum(
    target_centre,
    source_positions,
    dipole_moments,
    output="both",
)

field_error = relative_error(
    local_centre_result["H"][np.newaxis, :],
    direct_result["H"][np.newaxis, :],
)[0]
print(f"L2P field at target centre: {local_centre_result['H']}")
print(f"Direct field:               {direct_result['H']}")
print(f"Relative field error: {field_error:.6e}")

## What to observe

Multipole and local vectors use the same total-degree index ordering but play
different roles. Local degree zero approximates the potential at the target
centre, degree one its gradient, and higher degrees describe spatial variation
near that centre. Only a subsequent L2P evaluation turns them into particle
outputs.